# Librerías

In [1]:
# Librerías
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

# Módulos 
from astroquery.mpc import MPC
from math import ceil

# MPC API

In [2]:
# Verificar la conexión a internet.
def verificar_conexion():
    try:
        requests.get("http://www.google.com", timeout=5)
        print('✅ Conectado a internet.')
        return True
    
    except requests.ConnectionError:
        print('🛑 Sin conexión a internet.')
        return False

In [3]:
# Conexión con la API del MPC
nombre_cometa = r'C/2023 A3'
try: 
    if verificar_conexion():
        response_pagina = requests.get("https://data.minorplanetcenter.net/api/get-obs", json={"desigs": [nombre_cometa], "output_format":["ADES_DF"]})

        if response_pagina.ok:
            ades = response_pagina.json()[0]['ADES_DF']

except requests.ConnectionError:
    print(f'🛑 Se presentó un error al cargar la base de datos.\nError: {response_pagina.status_code}\n{response_pagina.content}')

✅ Conectado a internet.


In [4]:
# Creación del data frame Cometa
cometa_df = pd.DataFrame(ades)
cometa_df.__len__()

9465

In [5]:
# Numero de registros y variables sin filtrar la información
filas,columnas = cometa_df.shape
print(f'Registros: {filas}\nVariables: {columnas}')

Registros: 9465
Variables: 76


In [6]:
# Base de datos arrojada por la API
cometa_df.sample(5)

,Obstype,artsat,astcat,band,com,ctr,dec,decstar,delay,deltadec,...,subfrm,sys,trkid,trkmpc,trksub,trx,unctime,vel1,vel2,vel3
6983,optical,None,Gaia2,G,None,NaN,3.813332,None,None,None,...,None,None,00000IN4VZ,None,None,None,None,None,None,None
2783,optical,None,UCAC4,V,None,NaN,-6.49313,None,None,None,...,None,None,00000Hva6d,None,None,None,None,None,None,None
2089,optical,None,Gaia2,G,None,NaN,0.78622,None,None,None,...,None,None,00000HNC8x,CK23A030,CK23A030,None,None,None,None,None
1483,optical,None,ATLAS2,Sr,None,NaN,3.323873,None,None,None,...,None,None,00000HJ7b1,CK23A030,DEKAB810,None,None,None,None,None
1375,optical,None,Gaia3E,G,None,NaN,3.17594,None,None,None,...,None,None,00000HIT4p,CK23A030,ABC0152,None,None,None,None,None


In [7]:
# Tratamiento de los datos de interés
cometa_df['obs_date'] = pd.to_datetime(pd.to_datetime(cometa_df.obstime, format='mixed').dt.date)
cometa_df['magnitude'] = pd.to_numeric(cometa_df.mag)

In [8]:
# Creación del data frame curva de luz cruda
curva_de_luz_cruda_df = cometa_df[['obs_date', 'magnitude']].copy()
curva_de_luz_cruda_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9465 entries, 0 to 9464
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   obs_date   9465 non-null   datetime64[ns]
 1   magnitude  8837 non-null   float64       
dtypes: datetime64[ns](1), float64(1)
memory usage: 148.0 KB


In [9]:
# Numero de registros y variables con la información filtrada
filas,columnas = curva_de_luz_cruda_df.shape
print(f'Registros: {filas}\nVariables: {columnas}')

Registros: 9465
Variables: 2


In [10]:
# Data Frame de la curva de luz
curva_de_luz_cruda_df.sample(5)

,obs_date,magnitude
7412,2024-11-23,9.1
7357,2024-11-20,12.0
3308,2024-04-16,13.2
8557,2025-06-11,16.1
2357,2024-01-23,14.0


In [11]:
# Curva de luz cruda.
labels = {'obs_date':'Observation Date','magnitude':'Apparent total magnitude', 'obs_method_key' : 'Observation Method'}
fig = px.scatter(curva_de_luz_cruda_df, x='obs_date', y='magnitude', template= 'plotly_dark', labels= labels, title= f'Lightcurve of comet {nombre_cometa} MPC data')
fig.update_yaxes(autorange="reversed")
fig.show()

# Perihelio MPC API

In [13]:
# Conexión con la API de COBS para obtener el perihelio
try: 
    Link_MPC_API = f'https://data.minorplanetcenter.net/api/get-orb'

    if verificar_conexion():

        response = requests.get(Link_MPC_API, json={"desig": nombre_cometa})

        if response.ok:

            elementos_orbitales_mpc = response.json()[0]['mpc_orb']
            perihelio_mjd_mpc = elementos_orbitales_mpc[0]['COM']['coefficient_values'][-1]

            perihelio = (pd.Timestamp('1970-01-01') + pd.to_timedelta(perihelio_mjd_mpc - 40587, unit='D')).normalize()
            print('✅ Perihelio del cometa obtenido.')
    
except requests.ConnectionError:
    print(f'🛑 Se presentó un error al cargar la base de datos.\nError: {response.status_code}\n{response.content}')

✅ Conectado a internet.
✅ Perihelio del cometa obtenido.


# MPC efemerides API usando astroquery.

In [14]:
# Creación de data frame Ephemeris (conexión con la API del MPC)
efemerides_total = []

fecha_inicial = curva_de_luz_cruda_df.obs_date.min()
fecha_final = curva_de_luz_cruda_df.obs_date.max()
fechas = (fecha_final - fecha_inicial).days + 1

print('⌛ Conectando con la base de datos [MPC efemerides].')
for i in range(ceil(fechas/1441)):
    efemerides = MPC.get_ephemeris(nombre_cometa, start = str(fecha_inicial), number = 1441)  # type: ignore
    efemerides_ciclo_df = efemerides.to_pandas()
    efemerides_total.append(efemerides_ciclo_df)

    fecha_inicial = efemerides_ciclo_df.Date.max()

# Creación del data frame efemerides filtrada
efemerides_df = pd.concat(efemerides_total)
efemerides_df.columns = efemerides_df.columns.str.lower().str.replace(' ', '_')

efemerides_filtrada_df = efemerides_df[['date', 'delta','r', 'phase']].copy()
efemerides_filtrada_df = efemerides_filtrada_df.rename(columns = {'date':'obs_date'})
efemerides_filtrada_df['obs_date'] = pd.to_datetime(pd.to_datetime(efemerides_filtrada_df.obs_date).dt.date)
efemerides_filtrada_df.reset_index(inplace = True)

efemerides_filtrada_df

⌛ Conectando con la base de datos [MPC efemerides].


c:\Users\USER\AppData\Local\Programs\Python\Python313\Lib\site-packages\erfa\core.py:133: ErfaWarning:

ERFA function "dtf2d" yielded 423 of "dubious year (Note 6)"

c:\Users\USER\AppData\Local\Programs\Python\Python313\Lib\site-packages\erfa\core.py:133: ErfaWarning:

ERFA function "d2dtf" yielded 423 of "dubious year (Note 5)"



,index,obs_date,delta,r,phase
0,0,2022-04-09,9.059,9.906,3.2
1,1,2022-04-10,9.044,9.899,3.2
2,2,2022-04-11,9.029,9.891,3.1
3,3,2022-04-12,9.014,9.883,3.0
4,4,2022-04-13,9.000,9.876,3.0
...,...,...,...,...,...
2877,1436,2030-02-22,17.254,16.939,3.1
2878,1437,2030-02-23,17.248,16.944,3.2
2879,1438,2030-02-24,17.241,16.950,3.2
2880,1439,2030-02-25,17.234,16.956,3.2


In [15]:
# Info del data frame ephemeris
efemerides_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2882 entries, 0 to 1440
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   date           2882 non-null   datetime64[ns]
 1   ra             2882 non-null   float64       
 2   dec            2882 non-null   float64       
 3   delta          2882 non-null   float64       
 4   r              2882 non-null   float64       
 5   elongation     2882 non-null   float64       
 6   phase          2882 non-null   float64       
 7   v              2882 non-null   float64       
 8   proper_motion  2882 non-null   float64       
 9   direction      2882 non-null   float64       
dtypes: datetime64[ns](1), float64(9)
memory usage: 247.7 KB


In [16]:
# Dar a los datos el formato deseado
efemerides_df.date = pd.to_datetime(efemerides_df.date)
efemerides_df.date = pd.to_datetime(efemerides_df.date.dt.date)
efemerides_df.dtypes

date             datetime64[ns]
ra                      float64
dec                     float64
delta                   float64
r                       float64
elongation              float64
phase                   float64
v                       float64
proper_motion           float64
direction               float64
dtype: object

In [17]:
# Creación del data frame ephemeris filtrada
efemerides_filtrada_df = efemerides_df[['date', 'delta','r', 'phase']].copy()
efemerides_filtrada_df = efemerides_filtrada_df.rename(columns = {'date':'obs_date'})
efemerides_filtrada_df

,obs_date,delta,r,phase
0,2022-04-09,9.059,9.906,3.2
1,2022-04-10,9.044,9.899,3.2
2,2022-04-11,9.029,9.891,3.1
3,2022-04-12,9.014,9.883,3.0
4,2022-04-13,9.000,9.876,3.0
...,...,...,...,...
1436,2030-02-22,17.254,16.939,3.1
1437,2030-02-23,17.248,16.944,3.2
1438,2030-02-24,17.241,16.950,3.2
1439,2030-02-25,17.234,16.956,3.2


# Unión de las bases de datos.

In [18]:
# Unión de las bases de datos COBS y MPC
curva_de_luz_procesada_df = curva_de_luz_cruda_df.merge(efemerides_filtrada_df, on='obs_date')
curva_de_luz_procesada_df

,obs_date,magnitude,delta,r,phase
0,2022-04-09,23.04,9.059,9.906,3.2
1,2022-04-09,23.27,9.059,9.906,3.2
2,2022-04-23,22.97,8.870,9.800,2.4
3,2022-05-26,22.65,8.659,9.546,3.1
4,2022-05-26,22.29,8.659,9.546,3.1
...,...,...,...,...,...
9460,2025-11-20,20.30,6.392,5.788,7.4
9461,2026-05-23,18.60,6.879,7.477,6.5
9462,2026-05-23,18.60,6.879,7.477,6.5
9463,2026-05-27,19.40,6.882,7.511,6.3


In [19]:
# Información del data frame curva de lus procesada
curva_de_luz_procesada_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9465 entries, 0 to 9464
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   obs_date   9465 non-null   datetime64[ns]
 1   magnitude  8837 non-null   float64       
 2   delta      9465 non-null   float64       
 3   r          9465 non-null   float64       
 4   phase      9465 non-null   float64       
dtypes: datetime64[ns](1), float64(4)
memory usage: 369.9 KB


In [20]:
# Reducción de la magnitud aparente y calculo del Delta t
beta = 0

curva_de_luz_procesada_df['magnitud_reducida'] = (
    curva_de_luz_cruda_df['magnitude'] 
    - 5 * np.log10(curva_de_luz_procesada_df['delta'] * curva_de_luz_procesada_df['r'])
    - (beta * curva_de_luz_procesada_df['phase'])
    )

curva_de_luz_procesada_df['delta_t'] = (curva_de_luz_procesada_df.obs_date - perihelio) # type: ignore
curva_de_luz_procesada_df['delta_t'] = curva_de_luz_procesada_df.delta_t.apply(lambda delta_t: delta_t.days)

curva_de_luz_procesada_df

,obs_date,magnitude,delta,r,phase,magnitud_reducida,delta_t
0,2022-04-09,23.04,9.059,9.906,3.2,13.275107,-902
1,2022-04-09,23.27,9.059,9.906,3.2,13.505107,-902
2,2022-04-23,22.97,8.870,9.800,2.4,13.274252,-888
3,2022-05-26,22.65,8.659,9.546,3.1,13.063554,-855
4,2022-05-26,22.29,8.659,9.546,3.1,12.703554,-855
...,...,...,...,...,...,...,...
9460,2025-11-20,20.30,6.392,5.788,7.4,12.459174,419
9461,2026-05-23,18.60,6.879,7.477,6.5,10.043737,603
9462,2026-05-23,18.60,6.879,7.477,6.5,10.043737,603
9463,2026-05-27,19.40,6.882,7.511,6.3,10.832938,607


In [21]:
# Curva de luz reducida
labels = {'obs_date':'Observation Date','magnitud_reducida':'Apparent total magnitude processed', 'obs_method_key' : 'Observation Method'}
fig = px.scatter(curva_de_luz_procesada_df, x='obs_date', y='magnitud_reducida', template= 'plotly_dark', labels= labels, title=f'Reduced Lightcurve of comet {nombre_cometa}')
fig.update_yaxes(autorange="reversed")
fig.show()

In [22]:
# Curva de luz reducida
labels = {'delta_t':'t-Δt','magnitud_reducida':'Apparent total magnitude processed', 'obs_method_key' : 'Observation Method'}
fig = px.scatter(curva_de_luz_procesada_df, x='delta_t', y='magnitud_reducida', template= 'plotly_dark', labels= labels, title=f'Reduced Lightcurve of comet {nombre_cometa}')
fig.update_yaxes(autorange="reversed")
fig.show()

In [23]:
# Creación del data frame curva de luz promediada
numero_elementos_grupo = 9

curva_de_luz_promediada_df = curva_de_luz_procesada_df.copy()
curva_de_luz_promediada_df['promedio_movil'] = curva_de_luz_promediada_df.magnitud_reducida.rolling(window = numero_elementos_grupo).mean()
curva_de_luz_promediada_df

,obs_date,magnitude,delta,r,phase,magnitud_reducida,delta_t,promedio_movil
0,2022-04-09,23.04,9.059,9.906,3.2,13.275107,-902,NaN
1,2022-04-09,23.27,9.059,9.906,3.2,13.505107,-902,NaN
2,2022-04-23,22.97,8.870,9.800,2.4,13.274252,-888,NaN
3,2022-05-26,22.65,8.659,9.546,3.1,13.063554,-855,NaN
4,2022-05-26,22.29,8.659,9.546,3.1,12.703554,-855,NaN
...,...,...,...,...,...,...,...,...
9460,2025-11-20,20.30,6.392,5.788,7.4,12.459174,419,12.217825
9461,2026-05-23,18.60,6.879,7.477,6.5,10.043737,603,11.940320
9462,2026-05-23,18.60,6.879,7.477,6.5,10.043737,603,11.748262
9463,2026-05-27,19.40,6.882,7.511,6.3,10.832938,607,11.710559


In [24]:
# Curva de luz Promediada
labels = {'obs_date':'Observation Date','magnitud_reducida':'Max apparent total magnitude reduced', 'obs_method_key' : 'Observation Method'}
fig = px.scatter(curva_de_luz_promediada_df, x='obs_date', y='promedio_movil', template= 'plotly_dark', labels= labels, title= f'Average Lightcurve of comet {nombre_cometa}')
fig.update_yaxes(autorange="reversed")
fig.show()

In [25]:
# Curva de luz Promediada
labels = {'delta_t':'t - Δt','magnitud_reducida':'Max apparent total magnitude reduced', 'obs_method_key' : 'Observation Method'}
fig = px.scatter(curva_de_luz_promediada_df, x='delta_t', y='promedio_movil', template= 'plotly_dark', labels= labels, title= f'Average Lightcurve of comet {nombre_cometa}')
fig.update_yaxes(autorange="reversed")
fig.show()

# Curva de luz interna (Envolvente inferior) v1 (Promedio corrido -> agrupación)

In [26]:
# Creación del data frame curva de luz agrupada
curva_de_luz_interna_v1_df = curva_de_luz_promediada_df.groupby(by = 'obs_date').max()
curva_de_luz_interna_v1_df = curva_de_luz_interna_v1_df.reset_index()

curva_de_luz_interna_v1_df

,obs_date,magnitude,delta,r,phase,magnitud_reducida,delta_t,promedio_movil
0,2022-04-09,23.27,9.059,9.906,3.2,13.505107,-902,NaN
1,2022-04-23,22.97,8.870,9.800,2.4,13.274252,-888,NaN
2,2022-05-26,22.65,8.659,9.546,3.1,13.063554,-855,NaN
3,2022-06-02,22.69,8.655,9.492,3.6,13.116876,-848,12.983973
4,2022-06-08,22.80,8.662,9.446,4.1,13.235669,-842,12.872010
...,...,...,...,...,...,...,...,...
705,2025-11-15,20.70,6.299,5.739,7.8,12.909461,414,12.026778
706,2025-11-19,20.10,6.374,5.779,7.5,12.268676,418,12.157225
707,2025-11-20,20.30,6.392,5.788,7.4,12.459174,419,12.217825
708,2026-05-23,18.60,6.879,7.477,6.5,10.043737,603,11.940320


In [27]:
# Gráfica de luz interna
labels = {'obs_date':'Observation Date','magnitud_reducida':'Magnitude reduced'}
fig = px.scatter(curva_de_luz_interna_v1_df, x='obs_date', y='promedio_movil', template= 'plotly_dark', labels= labels, title=f'Min Averaged Lightcurve of comet {nombre_cometa}')
fig.update_traces(marker=dict(color='red', size=6, line=dict(width=1, color='DarkSlateGrey')))
fig.update_yaxes(autorange="reversed")
fig.show()

In [28]:
# Gráfica de luz interna
labels = {'delta_t':'t - Δt','magnitud_reducida':'Magnitude reduced'}
fig = px.scatter(curva_de_luz_interna_v1_df, x='delta_t', y='promedio_movil', template= 'plotly_dark', labels= labels, title=f'Min Averaged Lightcurve of comet {nombre_cometa}')
fig.update_traces(marker=dict(color='red', size=6, line=dict(width=1, color='DarkSlateGrey')))
fig.update_yaxes(autorange="reversed")
fig.show()

# Curva de luz Externa (Envolvente superior) v1 (Promedio corrido -> agrupación)

In [29]:
# Creación del data frame curva de luz agrupada
curva_de_luz_externa_v1_df = curva_de_luz_promediada_df.groupby(by = 'obs_date').min()
curva_de_luz_externa_v1_df = curva_de_luz_externa_v1_df.reset_index()
curva_de_luz_externa_v1_df

,obs_date,magnitude,delta,r,phase,magnitud_reducida,delta_t,promedio_movil
0,2022-04-09,23.04,9.059,9.906,3.2,13.275107,-902,NaN
1,2022-04-23,22.97,8.870,9.800,2.4,13.274252,-888,NaN
2,2022-05-26,22.25,8.659,9.546,3.1,12.663554,-855,NaN
3,2022-06-02,22.11,8.655,9.492,3.6,12.536876,-848,12.901947
4,2022-06-08,21.80,8.662,9.446,4.1,12.235669,-842,12.756612
...,...,...,...,...,...,...,...,...
705,2025-11-15,20.70,6.299,5.739,7.8,12.909461,414,11.861759
706,2025-11-19,19.90,6.374,5.779,7.5,12.068676,418,12.097557
707,2025-11-20,19.70,6.392,5.788,7.4,11.859174,419,12.171392
708,2026-05-23,18.60,6.879,7.477,6.5,10.043737,603,11.748262


In [30]:
# Gráfica de lus promediada
labels = {'obs_date':'Observation Date','magnitud_reducida':'Magnitude reduced'}
fig = px.scatter(curva_de_luz_externa_v1_df, x='obs_date', y='promedio_movil', template= 'plotly_dark', labels= labels, title=f'Max averaged Lightcurve of comet {nombre_cometa}')
fig.update_traces(marker=dict(color='yellow', size=6, line= dict(width=1, color='DarkSlateGrey')))
fig.update_yaxes(autorange="reversed")
fig.show()

In [31]:
# Gráfica de lus promediada
labels = {'delta_t':'t - Δt','magnitud_reducida':'Magnitude reduced'}
fig = px.scatter(curva_de_luz_externa_v1_df, x='delta_t', y='promedio_movil', template= 'plotly_dark', labels= labels, title=f'Max averaged Lightcurve of comet {nombre_cometa}')
fig.update_traces(marker=dict(color='yellow', size=6, line= dict(width=1, color='DarkSlateGrey')))
fig.update_yaxes(autorange="reversed")
fig.show()

# Curva de luz interna (Envolvente inferior) v2 (Agrupación  -> promedio corrido)

In [32]:
# Creación del data frame curva de luz agrupada
curva_de_luz_agrupada_max_v2_df = curva_de_luz_procesada_df.groupby(by = 'obs_date').max()
curva_de_luz_agrupada_max_v2_df = curva_de_luz_agrupada_max_v2_df.reset_index()
curva_de_luz_agrupada_max_v2_df

,obs_date,magnitude,delta,r,phase,magnitud_reducida,delta_t
0,2022-04-09,23.27,9.059,9.906,3.2,13.505107,-902
1,2022-04-23,22.97,8.870,9.800,2.4,13.274252,-888
2,2022-05-26,22.65,8.659,9.546,3.1,13.063554,-855
3,2022-06-02,22.69,8.655,9.492,3.6,13.116876,-848
4,2022-06-08,22.80,8.662,9.446,4.1,13.235669,-842
...,...,...,...,...,...,...,...
705,2025-11-15,20.70,6.299,5.739,7.8,12.909461,414
706,2025-11-19,20.10,6.374,5.779,7.5,12.268676,418
707,2025-11-20,20.30,6.392,5.788,7.4,12.459174,419
708,2026-05-23,18.60,6.879,7.477,6.5,10.043737,603


In [33]:
# Gráfica de luz interna
labels = {'obs_date':'Observation Date','magnitud_reducida':'Magnitude reduced'}
fig = px.scatter(curva_de_luz_agrupada_max_v2_df, x='obs_date', y='magnitud_reducida', template= 'plotly_dark', labels= labels, title=f'Min Lightcurve of comet {nombre_cometa}')
fig.update_yaxes(autorange="reversed")
fig.show()

In [34]:
# Creación del data frame curva de luz promediada
numero_elementos_grupo = 7

curva_de_luz_interna_v2_df = curva_de_luz_agrupada_max_v2_df.copy()
curva_de_luz_interna_v2_df['promedio_movil'] = curva_de_luz_interna_v2_df.magnitud_reducida.rolling(window = numero_elementos_grupo, center= True).mean()
curva_de_luz_interna_v2_df

,obs_date,magnitude,delta,r,phase,magnitud_reducida,delta_t,promedio_movil
0,2022-04-09,23.27,9.059,9.906,3.2,13.505107,-902,NaN
1,2022-04-23,22.97,8.870,9.800,2.4,13.274252,-888,NaN
2,2022-05-26,22.65,8.659,9.546,3.1,13.063554,-855,NaN
3,2022-06-02,22.69,8.655,9.492,3.6,13.116876,-848,13.131958
4,2022-06-08,22.80,8.662,9.446,4.1,13.235669,-842,12.742372
...,...,...,...,...,...,...,...,...
705,2025-11-15,20.70,6.299,5.739,7.8,12.909461,414,11.960893
706,2025-11-19,20.10,6.374,5.779,7.5,12.268676,418,12.061075
707,2025-11-20,20.30,6.392,5.788,7.4,12.459174,419,NaN
708,2026-05-23,18.60,6.879,7.477,6.5,10.043737,603,NaN


In [35]:
# Gráfica de luz interna
labels = {'obs_date':'Observation Date','magnitud_reducida':'Magnitude reduced'}
fig = px.scatter(curva_de_luz_interna_v2_df, x='obs_date', y='promedio_movil', template= 'plotly_dark', labels= labels, title=f'Min Averaged Lightcurve of comet {nombre_cometa}')
fig.update_traces(marker=dict(color='red', size=5, line=dict(width=1, color='DarkSlateGrey')))
fig.update_yaxes(autorange="reversed")
fig.show()

In [36]:
# Gráfica de luz interna
labels = {'delta_t':'t - Δt','magnitud_reducida':'Magnitude reduced'}
fig = px.scatter(curva_de_luz_interna_v2_df, x='delta_t', y='promedio_movil', template= 'plotly_dark', labels= labels, title=f'Min Averaged Lightcurve of comet {nombre_cometa}')
fig.update_traces(marker=dict(color='red', size=5, line=dict(width=1, color='DarkSlateGrey')))
fig.update_yaxes(autorange="reversed")
fig.show()

# Curva de luz Externa (Envolvente superior) v2 (Agrupación  -> promedio corrido)

In [37]:
# Creación del data frame curva de luz agrupada
curva_de_luz_agrupada_min_v2_df = curva_de_luz_procesada_df.groupby(by = 'obs_date').min()
curva_de_luz_agrupada_min_v2_df = curva_de_luz_agrupada_min_v2_df.reset_index()
curva_de_luz_agrupada_min_v2_df

,obs_date,magnitude,delta,r,phase,magnitud_reducida,delta_t
0,2022-04-09,23.04,9.059,9.906,3.2,13.275107,-902
1,2022-04-23,22.97,8.870,9.800,2.4,13.274252,-888
2,2022-05-26,22.25,8.659,9.546,3.1,12.663554,-855
3,2022-06-02,22.11,8.655,9.492,3.6,12.536876,-848
4,2022-06-08,21.80,8.662,9.446,4.1,12.235669,-842
...,...,...,...,...,...,...,...
705,2025-11-15,20.70,6.299,5.739,7.8,12.909461,414
706,2025-11-19,19.90,6.374,5.779,7.5,12.068676,418
707,2025-11-20,19.70,6.392,5.788,7.4,11.859174,419
708,2026-05-23,18.60,6.879,7.477,6.5,10.043737,603


In [38]:
# Gráfica de luz interna
labels = {'obs_date':'Observation Date','magnitud_reducida':'Magnitude reduced'}
fig = px.scatter(curva_de_luz_agrupada_min_v2_df, x='obs_date', y='magnitud_reducida', template= 'plotly_dark', labels= labels, title=f'Min Averaged Lightcurve of comet {nombre_cometa}')
fig.update_yaxes(autorange="reversed")
fig.show()

In [39]:
# Creación del data frame curva de luz promediada
numero_elementos_grupo = 7

curva_de_luz_externa_v2_df = curva_de_luz_agrupada_min_v2_df.copy()
curva_de_luz_externa_v2_df['promedio_movil'] = curva_de_luz_externa_v2_df.magnitud_reducida.rolling(window = numero_elementos_grupo, center= True).mean()
curva_de_luz_externa_v2_df

,obs_date,magnitude,delta,r,phase,magnitud_reducida,delta_t,promedio_movil
0,2022-04-09,23.04,9.059,9.906,3.2,13.275107,-902,NaN
1,2022-04-23,22.97,8.870,9.800,2.4,13.274252,-888,NaN
2,2022-05-26,22.25,8.659,9.546,3.1,12.663554,-855,NaN
3,2022-06-02,22.11,8.655,9.492,3.6,12.536876,-848,12.716244
4,2022-06-08,21.80,8.662,9.446,4.1,12.235669,-842,12.316657
...,...,...,...,...,...,...,...,...
705,2025-11-15,20.70,6.299,5.739,7.8,12.909461,414,11.646608
706,2025-11-19,19.90,6.374,5.779,7.5,12.068676,418,11.561075
707,2025-11-20,19.70,6.392,5.788,7.4,11.859174,419,NaN
708,2026-05-23,18.60,6.879,7.477,6.5,10.043737,603,NaN


In [40]:
# Gráfica de luz interna
labels = {'obs_date':'Observation Date','magnitud_reducida':'Magnitude reduced'}
fig = px.scatter(curva_de_luz_externa_v2_df, x='obs_date', y='promedio_movil', template= 'plotly_dark', labels= labels, title=f'Max Averaged Lightcurve of comet {nombre_cometa}')
fig.update_traces(marker=dict(color='yellow', size=5, line=dict(width=1, color='DarkSlateGrey')))
fig.update_yaxes(autorange="reversed")
fig.show()

In [41]:
# Gráfica de luz interna
labels = {'delta_t':'t - Δt','magnitud_reducida':'Magnitude reduced'}
fig = px.scatter(curva_de_luz_externa_v2_df, x='delta_t', y='promedio_movil', template= 'plotly_dark', labels= labels, title=f'Max Averaged Lightcurve of comet {nombre_cometa}')
fig.update_traces(marker=dict(color='yellow', size=5, line=dict(width=1, color='DarkSlateGrey')))
fig.update_yaxes(autorange="reversed")
fig.show()

# Curva de luz usando la mediada v1 (Mediana de cada día registrado).

In [42]:
# Creación del data frame curva de luz mediana v1
curva_de_luz_mediada_v1_df = curva_de_luz_procesada_df.groupby(by= 'obs_date').median(numeric_only= True)
curva_de_luz_mediada_v1_df = curva_de_luz_mediada_v1_df.reset_index()
curva_de_luz_mediada_v1_df

,obs_date,magnitude,delta,r,phase,magnitud_reducida,delta_t
0,2022-04-09,23.155,9.059,9.906,3.2,13.390107,-902.0
1,2022-04-23,22.970,8.870,9.800,2.4,13.274252,-888.0
2,2022-05-26,22.290,8.659,9.546,3.1,12.703554,-855.0
3,2022-06-02,22.200,8.655,9.492,3.6,12.626876,-848.0
4,2022-06-08,22.300,8.662,9.446,4.1,12.735669,-842.0
...,...,...,...,...,...,...,...
705,2025-11-15,20.700,6.299,5.739,7.8,12.909461,414.0
706,2025-11-19,20.000,6.374,5.779,7.5,12.168676,418.0
707,2025-11-20,20.000,6.392,5.788,7.4,12.159174,419.0
708,2026-05-23,18.600,6.879,7.477,6.5,10.043737,603.0


In [43]:
# Gráfica de luz interna
labels = {'obs_date':'Observation Date','magnitud_reducida':'Magnitude reduced'}
fig = px.scatter(curva_de_luz_mediada_v1_df, x='obs_date', y='magnitud_reducida', template= 'plotly_dark', labels= labels, title=f'Mediated Lightcurve of comet {nombre_cometa}')
fig.update_traces(marker=dict(color='aquamarine', size=6, line=dict(width=1, color='DarkSlateGrey')))
fig.update_yaxes(autorange="reversed")
fig.show()

In [44]:
# Gráfica de luz interna
labels = {'delta_t':'t - Δt','magnitud_reducida':'Magnitude reduced'}
fig = px.scatter(curva_de_luz_mediada_v1_df, x='delta_t', y='magnitud_reducida', template= 'plotly_dark', labels= labels, title=f'Mediated Lightcurve of comet {nombre_cometa}')
fig.update_traces(marker=dict(color='aquamarine', size=6, line=dict(width=1, color='DarkSlateGrey')))
fig.update_yaxes(autorange="reversed")
fig.show()

# Curva de luz usando la mediada v2 (Mediana de las dos curvas).

In [45]:
# Creación del data frame curva de luz mediana v2
curva_de_luz_mediada_v2_df = curva_de_luz_externa_v2_df.copy()
curva_de_luz_mediada_v2_df['mediana'] = (curva_de_luz_interna_v2_df['promedio_movil'] + curva_de_luz_externa_v2_df['promedio_movil'])/2
curva_de_luz_mediada_v2_df

,obs_date,magnitude,delta,r,phase,magnitud_reducida,delta_t,promedio_movil,mediana
0,2022-04-09,23.04,9.059,9.906,3.2,13.275107,-902,NaN,NaN
1,2022-04-23,22.97,8.870,9.800,2.4,13.274252,-888,NaN,NaN
2,2022-05-26,22.25,8.659,9.546,3.1,12.663554,-855,NaN,NaN
3,2022-06-02,22.11,8.655,9.492,3.6,12.536876,-848,12.716244,12.924101
4,2022-06-08,21.80,8.662,9.446,4.1,12.235669,-842,12.316657,12.529514
...,...,...,...,...,...,...,...,...,...
705,2025-11-15,20.70,6.299,5.739,7.8,12.909461,414,11.646608,11.803751
706,2025-11-19,19.90,6.374,5.779,7.5,12.068676,418,11.561075,11.811075
707,2025-11-20,19.70,6.392,5.788,7.4,11.859174,419,NaN,NaN
708,2026-05-23,18.60,6.879,7.477,6.5,10.043737,603,NaN,NaN


In [46]:
# Gráfica de luz interna
labels = {'obs_date':'Observation Date','magnitud_reducida':'Magnitude reduced'}
fig = px.scatter(curva_de_luz_mediada_v2_df, x='obs_date', y='mediana', template= 'plotly_dark', labels= labels, title=f'Mediated Lightcurve of comet {nombre_cometa}')
fig.update_traces(marker=dict(color='aquamarine', size=6, line=dict(width=1, color='DarkSlateGrey')))
fig.update_yaxes(autorange="reversed")
fig.show()

In [47]:
# Gráfica de luz interna
labels = {'delta_t':'t - Δt','magnitud_reducida':'Magnitude reduced'}
fig = px.scatter(curva_de_luz_mediada_v2_df, x='delta_t', y='mediana', template= 'plotly_dark', labels= labels, title=f'Mediated Lightcurve of comet {nombre_cometa}')
fig.update_traces(marker=dict(color='aquamarine', size=6, line=dict(width=1, color='DarkSlateGrey')))
fig.update_yaxes(autorange="reversed")
fig.show()

# Comparación de las curvas de luz v1 (Promedio corrido -> agrupación)

In [48]:
# Gráfica de luz promediada
labels = {'obs_date':'Observation Date','magnitud_reducida':'Magnitude reduced'}
fig = go.Figure()
fig.add_trace(go.Scatter(x=curva_de_luz_externa_v1_df.obs_date, y=curva_de_luz_externa_v1_df.promedio_movil, mode='markers', name='Envolvente', marker=dict(color='yellow', line=dict(width=1, color='DarkSlateGrey'))))
fig.add_trace(go.Scatter(x=curva_de_luz_interna_v1_df.obs_date, y=curva_de_luz_interna_v1_df.promedio_movil, mode='markers', name='Núcleo', marker=dict(color='red', line=dict(width=1, color='DarkSlateGrey'))))
fig.add_trace(go.Scatter(x=curva_de_luz_mediada_v1_df.obs_date, y=curva_de_luz_mediada_v1_df.magnitud_reducida, mode='markers', name='Mediana', marker=dict(color='aquamarine', line=dict(width=1, color='DarkSlateGrey'))))
# fig.add_trace(go.Scatter(x=curva_de_luz_externa_v2_df.obs_date, y=curva_de_luz_externa_v2_df.promedio_movil, mode='markers', name='Envolvente_v2', marker=dict(color='green', line=dict(width=1, color='DarkSlateGrey'))))
# fig.add_trace(go.Scatter(x=curva_de_luz_interna_v1_df.obs_date, y=curva_de_luz_interna_v2_df.promedio_movil, mode='markers', name='Núcleo_v2', marker=dict(color='blue', line=dict(width=1, color='DarkSlateGrey'))))
# fig.add_trace(go.Scatter(x=curva_de_luz_mediada_v1_df.obs_date, y=curva_de_luz_mediada_v2_df.mediana, mode='markers', name='Mediana_v2', marker=dict(color='aquamarine', line=dict(width=1, color='DarkSlateGrey'))))
fig.update_layout(template='plotly_dark')
fig.update_yaxes(autorange="reversed")
fig.update_layout(template='plotly_dark', xaxis_title='Observation Date', yaxis_title='Averaged Magnitude', title = f'Max/Min Averaged Lightcurve of comet {nombre_cometa}')
fig.show()

In [49]:
# Gráfica de luz promediada
fig = go.Figure()
fig.add_trace(go.Scatter(x=curva_de_luz_externa_v1_df.delta_t, y=curva_de_luz_externa_v1_df.promedio_movil, mode='markers', name='Envolvente', marker=dict(color='yellow', line=dict(width=1, color='DarkSlateGrey'))))
fig.add_trace(go.Scatter(x=curva_de_luz_interna_v1_df.delta_t, y=curva_de_luz_interna_v1_df.promedio_movil, mode='markers', name='Núcleo', marker=dict(color='red', line=dict(width=1, color='DarkSlateGrey'))))
fig.add_trace(go.Scatter(x=curva_de_luz_mediada_v1_df.delta_t, y=curva_de_luz_mediada_v1_df.magnitud_reducida, mode='markers', name='Mediana', marker=dict(color='aquamarine', line=dict(width=1, color='DarkSlateGrey'))))
fig.update_layout(template='plotly_dark')
fig.update_yaxes(autorange="reversed")
fig.update_layout(template='plotly_dark', xaxis_title='t - Δt', yaxis_title='Averaged Magnitude', title = f'Max/Min Averaged Lightcurve of comet {nombre_cometa}')
fig.show()

# Comparación de las curvas de luz v2 (Agrupación -> promedio corrido)

In [50]:
# Gráfica de luz promediada
fig = go.Figure()
fig.add_trace(go.Scatter(x=curva_de_luz_agrupada_min_v2_df.obs_date, y=curva_de_luz_agrupada_min_v2_df.magnitud_reducida, mode='markers', name='máximo diario', marker=dict(color="#fa00e9", line=dict(width=1, color='DarkSlateGrey'))))
fig.add_trace(go.Scatter(x=curva_de_luz_agrupada_max_v2_df.obs_date, y=curva_de_luz_agrupada_max_v2_df.magnitud_reducida, mode='markers', name='minimo diario', marker=dict(color="#02FA61", line=dict(width=1, color='DarkSlateGrey'))))
fig.add_trace(go.Scatter(x=curva_de_luz_externa_v2_df.obs_date, y=curva_de_luz_externa_v2_df.promedio_movil, mode='markers', name='Envolvente', marker=dict(color='yellow', line=dict(width=1, color='DarkSlateGrey'))))
fig.add_trace(go.Scatter(x=curva_de_luz_interna_v2_df.obs_date, y=curva_de_luz_interna_v2_df.promedio_movil, mode='markers', name='Núcleo', marker=dict(color='red', line=dict(width=1, color='DarkSlateGrey'))))
# fig.add_trace(go.Scatter(x=curva_de_luz_interna_v2_df.obs_date, y=curva_de_luz_mediada_v2_df.mediana, mode='markers', name='Mediana', marker=dict(color='aquamarine', line=dict(width=1, color='DarkSlateGrey'))))

fig.update_layout(template='plotly_dark')
fig.update_yaxes(autorange="reversed")
fig.update_layout(template='plotly_dark', xaxis_title='Observation Date', yaxis_title='Averaged Magnitude', title = f'Max/Min Averaged Lightcurve of comet {nombre_cometa}')

fig.show()

# Envolvente superior calculada con percentiles v3

In [51]:
# Creación del data frame curva de luz agrupada
curva_de_luz_agrupada_min_v3_df = curva_de_luz_procesada_df.groupby(by = 'obs_date').min()
curva_de_luz_agrupada_min_v3_df = curva_de_luz_agrupada_min_v3_df.reset_index()
curva_de_luz_agrupada_min_v3_df

,obs_date,magnitude,delta,r,phase,magnitud_reducida,delta_t
0,2022-04-09,23.04,9.059,9.906,3.2,13.275107,-902
1,2022-04-23,22.97,8.870,9.800,2.4,13.274252,-888
2,2022-05-26,22.25,8.659,9.546,3.1,12.663554,-855
3,2022-06-02,22.11,8.655,9.492,3.6,12.536876,-848
4,2022-06-08,21.80,8.662,9.446,4.1,12.235669,-842
...,...,...,...,...,...,...,...
705,2025-11-15,20.70,6.299,5.739,7.8,12.909461,414
706,2025-11-19,19.90,6.374,5.779,7.5,12.068676,418
707,2025-11-20,19.70,6.392,5.788,7.4,11.859174,419
708,2026-05-23,18.60,6.879,7.477,6.5,10.043737,603


In [58]:
curva_de_luz_interna_v3_df = curva_de_luz_agrupada_min_v3_df.copy()
curva_de_luz_interna_v3_df['percentil_movil'] = curva_de_luz_agrupada_min_v3_df.magnitud_reducida.rolling(window=3, center=True).apply(
    lambda x: np.percentile(x, 0), raw=True)
curva_de_luz_externa_v2_df

,obs_date,magnitude,delta,r,phase,magnitud_reducida,delta_t,promedio_movil
0,2022-04-09,23.04,9.059,9.906,3.2,13.275107,-902,NaN
1,2022-04-23,22.97,8.870,9.800,2.4,13.274252,-888,NaN
2,2022-05-26,22.25,8.659,9.546,3.1,12.663554,-855,NaN
3,2022-06-02,22.11,8.655,9.492,3.6,12.536876,-848,12.716244
4,2022-06-08,21.80,8.662,9.446,4.1,12.235669,-842,12.316657
...,...,...,...,...,...,...,...,...
705,2025-11-15,20.70,6.299,5.739,7.8,12.909461,414,11.646608
706,2025-11-19,19.90,6.374,5.779,7.5,12.068676,418,11.561075
707,2025-11-20,19.70,6.392,5.788,7.4,11.859174,419,NaN
708,2026-05-23,18.60,6.879,7.477,6.5,10.043737,603,NaN


In [59]:
# Gráfica de luz promediada
fig = go.Figure()
fig.add_trace(go.Scatter(x=curva_de_luz_procesada_df.obs_date, y=curva_de_luz_procesada_df.magnitud_reducida, mode='markers', name='magnitud reducida', marker=dict(line=dict(width=1, color='DarkSlateGrey'))))
fig.add_trace(go.Scatter(x=curva_de_luz_agrupada_min_v3_df.obs_date, y=curva_de_luz_agrupada_min_v3_df.magnitud_reducida, mode='markers', name='máximo diario', marker=dict(color="#fa00e9", line=dict(width=1, color='DarkSlateGrey'))))
fig.add_trace(go.Scatter(x=curva_de_luz_interna_v3_df.obs_date, y=curva_de_luz_interna_v3_df.percentil_movil, mode='markers', name='Envolvente Percentil', marker=dict(color="#02c7fe", line=dict(width=1, color='DarkSlateGrey'))))
fig.add_trace(go.Scatter(x=curva_de_luz_interna_v2_df.obs_date, y=curva_de_luz_externa_v2_df.promedio_movil, mode='markers', name='Envolvente Promedio', marker=dict(color='yellow', line=dict(width=1, color='DarkSlateGrey'))))

fig.update_layout(template='plotly_dark')
fig.update_yaxes(autorange="reversed")
fig.update_layout(template='plotly_dark', xaxis_title='Observation Date', yaxis_title='Averaged Magnitude', title = f'Max/Min Averaged Lightcurve of comet {nombre_cometa}')

fig.show()

# Guardar datos.

In [60]:
curva_de_luz_procesada_df.to_csv(r'Bases_de_datos/curva_de_luz_procesada_MPC.txt', index=False)
curva_de_luz_interna_v2_df.to_csv(r'Bases_de_datos/curva_de_luz_interna_MPC.txt', index=False)
curva_de_luz_externa_v2_df.to_csv(r'Bases_de_datos/curva_de_luz_externa_MPC.txt', index=False)